In [ ]:
pip install pruna pillow

In [ ]:
from pruna import PrunaModel, smash, SmashConfig

# Load base model
base_flux = PrunaModel.from_pretrained(
    "black-forest-labs/FLUX.2-klein-4B"
)

# Smash configuration (adjust as needed)
smash_cfg = SmashConfig([
    "deepcache",
    "compiler(torch_compile)"
])

# Apply smash
flux = smash(base_flux, smash_cfg)

# Optional: save optimized artifact
flux.save_pretrained("flux_klein_4b_smashed")

In [ ]:
import dspy
from dspy import settings

settings.configure(
    lm=dspy.HFModel(
        model="HuggingFaceTB/SmolVLM-Instruct",
        temperature=0.0
    )
)

In [ ]:
class PromptSignature(dspy.Signature):
    subject: str = dspy.InputField()
    optimized_prompt: str = dspy.OutputField(
        desc="Highly descriptive image generation prompt"
    )

prompt_module = dspy.Predict(PromptSignature)

In [ ]:
class ImageGrade(dspy.Signature):
    prompt: str = dspy.InputField()
    image: object = dspy.InputField()

    prompt_adherence: float = dspy.OutputField()
    aesthetic_quality: float = dspy.OutputField()
    text_correctness: float = dspy.OutputField()

In [ ]:
def grading_metric(example, pred, trace=None):
    return (
        0.5 * pred.prompt_adherence +
        0.3 * pred.aesthetic_quality +
        0.2 * pred.text_correctness
    )

In [ ]:
from dspy import MIPROv2

optimizer = MIPROv2(
    metric=grading_metric,
    auto="medium"
)

In [ ]:
subjects = [
    "Minimalist coffee brand poster",
    "Luxury chocolate advertisement",
    "Eco-friendly skincare packaging design",
    "Modern tech startup hero banner",
    "Futuristic sportswear campaign"
]

In [ ]:
import os
from datasets import Dataset
from PIL import Image

os.makedirs("generated_images", exist_ok=True)

records = []

for subject in subjects:
    
    # ----------------------------
    # DSPy Prompt Search
    # ----------------------------
    compiled_prompt_module = optimizer.compile(
        prompt_module,
        trainset=[{"subject": subject}]
    )
    
    optimized = compiled_prompt_module(subject=subject)
    optimized_prompt = optimized.optimized_prompt
    
    candidates = []
    
    # ----------------------------
    # Generate Multiple Candidates
    # ----------------------------
    for i in range(4):
        
        result = flux.generate(
            positivePrompt=optimized_prompt
        )
        
        image = result.images[0]
        
        # ----------------------------
        # SmolVLM Grading
        # ----------------------------
        grade = vlm_grader(
            prompt=optimized_prompt,
            image=image
        )
        
        score = grading_metric(None, grade)
        
        candidates.append({
            "image": image,
            "score": score,
            "grades": {
                "prompt_adherence": grade.prompt_adherence,
                "aesthetic_quality": grade.aesthetic_quality,
                "text_correctness": grade.text_correctness,
            }
        })
    
    # ----------------------------
    # Top-k Selection
    # ----------------------------
    candidates.sort(key=lambda x: x["score"], reverse=True)
    top_candidates = candidates[:2]
    
    for idx, item in enumerate(top_candidates):
        image_path = f"generated_images/{subject.replace(' ','_')}_{idx}.png"
        item["image"].save(image_path)
        
        records.append({
            "subject": subject,
            "optimized_prompt": optimized_prompt,
            "image_path": image_path,
            "score": item["score"],
            **item["grades"]
        })